## Importing and Initial Variables

In [1]:
# from todoist_api_python.api import TodoistAPI
import pandas as pd
from utils import credentials as cd, config as config
from dateutil import parser as dateparser
import requests
from datetime import datetime
from pathlib import Path

COLUMN_TEMPLATE = config.TODOIST_COLUMNS
FILENAME = "my-energysystem-tasks_raw.csv"
mostrecentdate = None
df = None
data_path = Path(Path.cwd(),r"data/",FILENAME)
baseurl = "https://api.todoist.com/api/v1/"
get_tasks = "tasks/completed/by_completion_date/"
get_projects = "projects/"
date_format = r"%Y/%m/%d"
headers = {"Authorization": f"Bearer {cd.api_key}"}

request_body = {
    "since" : None,
    "until" : None,
    "limit": "200"
}

In [2]:
Path.exists(data_path)

False

## Functions

In [3]:
def isoToString(date, format):
    input_date = dateparser.parse(date)
    return input_date.strftime(format=format)

def column_is_type(df):
    return df.transform(lambda x: x.apply(type)).drop_duplicates().iloc[0]

def split_column(data,split,key):
    test = data[[key,split]]
    test = test[test.iloc[:, 1].notna()]
    # print(column_is_type(test.iloc[:,1]))
    if column_is_type(test.iloc[:,1]) is dict:
        unnest = test[split].apply(pd.Series)
        source_names = unnest.columns.tolist()
        updated_names = [split.capitalize() + part.capitalize() for part in source_names]

        rename_zip = zip(source_names, updated_names)
        rename_dict = dict(rename_zip)

        renamed = unnest.rename(rename_dict, axis=1)
        output = test.join(renamed)
        return output

def todistRequest(request, param, header):
    items = []
    result_type = "items"
    cursor = None
    if 'project' in request:
            result_type = "results"
    url = baseurl + request
    while True:
        params = param
        if cursor:
            params["cursor"] = cursor

        response = requests.get(url, params=params, headers=header)
        response.raise_for_status()
        data = response.json()

        items.extend(data.get(result_type))

        cursor = data.get("next_cursor")
        if not cursor:
            break
    return items

## Check for existing archive of tasks

In [4]:
# Path.read_bytes(data_path)
if Path.exists(data_path) is False:
    print("file not found.\npull from earliest month start.")
else:
    print("file found. Opening to collect last task date")
    df = pd.read_csv(data_path).infer_objects()
    mostrecentdate = df['completed_at'].max()
    

file not found.
pull from earliest month start.


### Grab the last completed date as a starting point to pull more tasks

In [5]:
end_date = datetime.now()

if mostrecentdate is None:
    print("No most recent date found")
    start_date = datetime.now()
    start_date = datetime.replace(start_date,day=1)
else:    
    mostrecentdate = dateparser.parse(mostrecentdate)
    start_date = mostrecentdate
    
iso_start_date = start_date.isoformat()
iso_end_date = end_date.isoformat()
    # datetime.strptime("5/1/2026","%m/%d/%Y")
# end_date = datetime.strptime("5/18/2026","%m/%d/%Y")



No most recent date found


## Collect Project and Tasks

In [6]:
request_body['since'] = start_date
request_body['until'] = end_date

In [7]:
project_request = request_body

projects = todistRequest(get_projects, project_request, headers)
proj_df = pd.DataFrame(projects)
proj_df = proj_df[['id','name']]
proj_df.rename(columns={"name":"ProjectName","id":"ProjectID"},inplace=True)
proj_df

,ProjectID,ProjectName
0,6Mw9JcR2gjPR3Xjm,Inbox
1,6XQ4r58cjq52mMvR,THIS WEEK
2,6XQ4rFJqXG3g4VH9,NEXT WEEK
3,6XQ4rFr9QRhXRMj7,THIS MONTH
4,6XQ4rGM3VhmcrvWQ,NEXT MONTH
5,6XQ4rH982p3r3H2g,LONG-TERM | ON HOLD
6,6gRgWCHV3JgFVpxH,PLANNING
7,6gRgWG8Hh74482vX,ROUTINES


In [8]:
items = todistRequest(get_tasks,request_body,headers)

## Build Task Table

In [9]:
task_table = pd.DataFrame(items)

if df is not None:
    task_table = pd.concat([task_table,df])

task_table = task_table.reset_index(drop=True)
# task_table


In [10]:
id_column = (task_table.columns.get_loc("id"), "id")

## Split due date iterable into unique columns

In [11]:
date_columns = split_column(data=task_table,split='due',key=id_column[1])

# date_columns
if date_columns is not None:
    combined_df = pd.merge(left=task_table, right=date_columns, on=id_column[1],how='left')
# combined_df

In [12]:
dates_to_convert = ['added_at', 'completed_at','updated_at']

combined_df[dates_to_convert] = combined_df[dates_to_convert].apply(lambda row: [isoToString(rowItem,date_format) for rowItem in row])

In [13]:
combined_df

,added_at,added_by_uid,assigned_by_uid,checked,child_order,completed_at,completed_by_uid,content,day_order,deadline,...,responsible_uid,section_id,updated_at,user_id,due_y,DueDate,DueIs_recurring,DueLang,DueString,DueTimezone
0,2026/05/22,43030779,None,True,22,2026/05/22,43030779,brainstorm other folks for a wedding guest list,-1,None,...,None,NaN,2026/05/22,43030779,"{'date': '2026-05-22', 'is_recurring': False, ...",2026-05-22,False,en,May 22,None
1,2026/05/22,43030779,None,True,21,2026/05/22,43030779,research some wedding venues in Minneapolis,-1,None,...,None,NaN,2026/05/22,43030779,"{'date': '2026-05-22', 'is_recurring': False, ...",2026-05-22,False,en,May 22,None
2,2026/05/22,43030779,None,True,21,2026/05/22,43030779,Make a grocery list,-1,None,...,None,NaN,2026/05/22,43030779,"{'date': '2026-05-22', 'is_recurring': False, ...",2026-05-22,False,en,May 22,None
3,2026/05/22,43030779,None,True,21,2026/05/22,43030779,spend time coding for myself,-1,None,...,None,NaN,2026/05/22,43030779,"{'date': '2026-05-22', 'is_recurring': False, ...",2026-05-22,False,en,May 22,None
4,2026/05/22,43030779,None,True,23,2026/05/22,43030779,Call Lowe's about replacement fridge door,-1,None,...,None,NaN,2026/05/22,43030779,"{'date': '2026-05-21', 'is_recurring': False, ...",2026-05-21,False,en,May 21,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,2026/05/02,43030779,None,True,17,2026/05/02,43030779,Clean up mouse and droppings behind stove,3,None,...,None,NaN,2026/05/02,43030779,"{'date': '2026-05-02', 'is_recurring': False, ...",2026-05-02,False,en,May 2,None
73,2026/05/02,43030779,None,True,16,2026/05/02,43030779,pick up some beer from central ave liquor,2,None,...,None,NaN,2026/05/02,43030779,"{'date': '2026-05-02', 'is_recurring': False, ...",2026-05-02,False,en,May 2,None
74,2026/05/02,43030779,None,True,15,2026/05/02,43030779,get bike tire repaired at recovery bike shop,1,None,...,None,NaN,2026/05/02,43030779,"{'date': '2026-05-02', 'is_recurring': False, ...",2026-05-02,False,en,May 2,None
75,2026/05/02,43030779,None,True,15,2026/05/02,43030779,training with Dan at Fish Lake Park,1,None,...,None,NaN,2026/05/02,43030779,"{'date': '2026-05-02', 'is_recurring': False, ...",2026-05-02,False,en,May 2,None


In [15]:
proj_df

,ProjectID,ProjectName
0,6Mw9JcR2gjPR3Xjm,Inbox
1,6XQ4r58cjq52mMvR,THIS WEEK
2,6XQ4rFJqXG3g4VH9,NEXT WEEK
3,6XQ4rFr9QRhXRMj7,THIS MONTH
4,6XQ4rGM3VhmcrvWQ,NEXT MONTH
5,6XQ4rH982p3r3H2g,LONG-TERM | ON HOLD
6,6gRgWCHV3JgFVpxH,PLANNING
7,6gRgWG8Hh74482vX,ROUTINES


In [16]:
full_data = pd.merge(left=combined_df, right=proj_df,how='left',left_on="project_id",right_on='ProjectID')
full_data
# proj_df

,added_at,added_by_uid,assigned_by_uid,checked,child_order,completed_at,completed_by_uid,content,day_order,deadline,...,updated_at,user_id,due_y,DueDate,DueIs_recurring,DueLang,DueString,DueTimezone,ProjectID,ProjectName
0,2026/05/22,43030779,None,True,22,2026/05/22,43030779,brainstorm other folks for a wedding guest list,-1,None,...,2026/05/22,43030779,"{'date': '2026-05-22', 'is_recurring': False, ...",2026-05-22,False,en,May 22,None,6XQ4r58cjq52mMvR,THIS WEEK
1,2026/05/22,43030779,None,True,21,2026/05/22,43030779,research some wedding venues in Minneapolis,-1,None,...,2026/05/22,43030779,"{'date': '2026-05-22', 'is_recurring': False, ...",2026-05-22,False,en,May 22,None,6XQ4r58cjq52mMvR,THIS WEEK
2,2026/05/22,43030779,None,True,21,2026/05/22,43030779,Make a grocery list,-1,None,...,2026/05/22,43030779,"{'date': '2026-05-22', 'is_recurring': False, ...",2026-05-22,False,en,May 22,None,6XQ4r58cjq52mMvR,THIS WEEK
3,2026/05/22,43030779,None,True,21,2026/05/22,43030779,spend time coding for myself,-1,None,...,2026/05/22,43030779,"{'date': '2026-05-22', 'is_recurring': False, ...",2026-05-22,False,en,May 22,None,6XQ4r58cjq52mMvR,THIS WEEK
4,2026/05/22,43030779,None,True,23,2026/05/22,43030779,Call Lowe's about replacement fridge door,-1,None,...,2026/05/22,43030779,"{'date': '2026-05-21', 'is_recurring': False, ...",2026-05-21,False,en,May 21,None,6XQ4r58cjq52mMvR,THIS WEEK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,2026/05/02,43030779,None,True,17,2026/05/02,43030779,Clean up mouse and droppings behind stove,3,None,...,2026/05/02,43030779,"{'date': '2026-05-02', 'is_recurring': False, ...",2026-05-02,False,en,May 2,None,6XQ4r58cjq52mMvR,THIS WEEK
73,2026/05/02,43030779,None,True,16,2026/05/02,43030779,pick up some beer from central ave liquor,2,None,...,2026/05/02,43030779,"{'date': '2026-05-02', 'is_recurring': False, ...",2026-05-02,False,en,May 2,None,6XQ4r58cjq52mMvR,THIS WEEK
74,2026/05/02,43030779,None,True,15,2026/05/02,43030779,get bike tire repaired at recovery bike shop,1,None,...,2026/05/02,43030779,"{'date': '2026-05-02', 'is_recurring': False, ...",2026-05-02,False,en,May 2,None,6XQ4r58cjq52mMvR,THIS WEEK
75,2026/05/02,43030779,None,True,15,2026/05/02,43030779,training with Dan at Fish Lake Park,1,None,...,2026/05/02,43030779,"{'date': '2026-05-02', 'is_recurring': False, ...",2026-05-02,False,en,May 2,None,6XQ4r58cjq52mMvR,THIS WEEK


## Drop Raw data & Unneeded columns

In [ ]:
final_df = combined_df.drop(columns=COLUMN_TEMPLATE['drop'], errors="ignore")

final_df = final_df.dropna(axis=1,how="all")
final_df.info()

In [ ]:
final_df